In [1]:
! pip install -U langchain langchain-openai langchain-community langchain-chroma langchain-huggingface chromadb pypdf python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os

from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

C:\Users\AMIT\AppData\Local\Temp\ipykernel_30836\1576796905.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv()

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

print("API key loaded:", bool(openrouter_api_key))

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_api_key
)


API key loaded: True


In [4]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "financial_policy_guidelines_and_example.pdf"

loader = PyPDFLoader(pdf_path)

documents = loader.load()

print("PDF loaded successfully")
print("Number of pages:", len(documents))

PDF loaded successfully
Number of pages: 3


In [5]:
print(documents)

[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 13.1 (Windows)', 'creationdate': '2018-12-27T12:58:46-06:00', 'moddate': '2018-12-27T12:58:46-06:00', 'trapped': '/False', 'source': 'financial_policy_guidelines_and_example.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='Copyright © Propel Nonprofits  |  1 SE Main St, Suite 600, Minneapolis, MN 55414  |  612.249.6700  |  propelnonprofits.org\nNonprofit Financial Policy \nGuidelines and Example\nDeveloping and adopting a written financial policy is a valuable practice \nfor any nonprofit organization, no matter how small or large. Financial policies clarify the roles, authority, \nand responsibilities for essential financial management activities and decisions. In the absence of an \nadopted policy, staff and board members are likely to operate under a set of assumptions that may or may \nnot be accurate or productive. If the idea of creating a financial policy seems daunting, t

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Documents split successfully")
print("Total chunks:", len(chunks))

Documents split successfully
Total chunks: 12


In [7]:
print(chunks[0].page_content)

Copyright © Propel Nonprofits  |  1 SE Main St, Suite 600, Minneapolis, MN 55414  |  612.249.6700  |  propelnonprofits.org
Nonprofit Financial Policy 
Guidelines and Example
Developing and adopting a written financial policy is a valuable practice 
for any nonprofit organization, no matter how small or large. Financial policies clarify the roles, authority, 
and responsibilities for essential financial management activities and decisions. In the absence of an 
adopted policy, staff and board members are likely to operate under a set of assumptions that may or may 
not be accurate or productive. If the idea of creating a financial policy seems daunting, these guidelines for 
policy development and this basic example may be helpful. Even though there may be occasional deficits, 
or periods of tight cash flow, the following characteristics are good signs that your organization will be 
financially healthy over the long-term.
more detail. The most important action is to create


In [8]:
embedding = client.embeddings.create(
    model="liquid/lfm-2.5-embedding-350m:free",
    input=[chunk.page_content for chunk in chunks]
)

print("Embeddings generated successfully")
print("Total embeddings:", len(embedding.data))
print("Embedding dimension:", len(embedding.data[0].embedding))

Embeddings generated successfully
Total embeddings: 12
Embedding dimension: 1024


In [9]:
embedding = client.embeddings.create(
    model="liquid/lfm-2.5-embedding-350m:free",
    input=[chunk.page_content for chunk in chunks]
)

print("Embeddings generated successfully")
print("Number of embeddings:", len(embedding.data))
print("Embedding dimension:", len(embedding.data[0].embedding))

Embeddings generated successfully
Number of embeddings: 12
Embedding dimension: 1024


In [10]:
from langchain_core.embeddings import Embeddings

class OpenRouterEmbeddings(Embeddings):

    def embed_documents(self, texts):
        response = client.embeddings.create(
            model="liquid/lfm-2.5-embedding-350m:free",
            input=texts
        )
        return [item.embedding for item in response.data]

    def embed_query(self, text):
        response = client.embeddings.create(
            model="liquid/lfm-2.5-embedding-350m:free",
            input=text
        )
        return response.data[0].embedding


embeddings = OpenRouterEmbeddings()

print("OpenRouter embeddings ready")

OpenRouter embeddings ready


In [11]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS vector store created successfully")

FAISS vector store created successfully


In [12]:
vectorstore.save_local("faiss_index")

print("FAISS index saved successfully")

FAISS index saved successfully


In [13]:
# Create a retriever using similarity search

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("Similarity search retriever created successfully")

Similarity search retriever created successfully


In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_api_key,
    model="openai/gpt-4o-mini"
)

query = input("Enter your query: ")

retrieved_docs = retriever.invoke(query)

context = "\n\n".join(
    doc.page_content
    for doc in retrieved_docs
)

prompt = f"""
Answer the question using only the provided context.

Context:
{context}

Question:
{query}

If the answer is not present in the context, say:
"I could not find this information in the provided documents."
"""

response = llm.invoke(prompt)


In [ ]:
answer=response.content

In [ ]:
type (answer)

str

In [ ]:
print(answer)

Based on the provided context, every financial policy needs to address five areas:

1. **Assignment of authority** for necessary and regular financial actions and decisions, which may include delegation of some authority to staff leaders

2. **Policy statement on conflicts of interest** or insider transactions

3. **Clear authority to spend funds**, including approval, check signing, and payroll

4. **Clear assignment of authority to enter into contracts**

5. **Clear responsibility for maintaining accurate financial records**

According to the context, the purpose of the financial policy is to describe and document how the board wants financial management activities to be carried out.
